# NGS pipeline · reads → variants → sickle cell

Real reads from a real sickle-cell carrier (1000 Genomes sample HG02666, GWD population, confirmed *HBB* rs334 heterozygote) aligned to a real human GRCh37 reference. Every cell runs real Linux binaries.

**Pipeline** · FASTQ → QC → trim → align → BAM → pileup → variant call → interpret

**Data source** · low-coverage Illumina reads streamed from the [1000 Genomes phase 3 BAM](https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/phase3/data/HG02666/alignment/) at chr11:5,246,001–5,250,000 (HBB region) — pre-extracted to a 97 KB FASTQ for speed.

Click **Runtime → Run all**, or run cells one by one with **Shift+Enter**.

---
## 0 · Install tools

In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y bwa samtools bcftools fastqc fastp 2>&1 | tail -3
echo
echo "=== installed versions ==="
bwa 2>&1      | head -3 | tail -1
samtools  --version | head -1
bcftools  --version | head -1
fastp     --version 2>&1 | head -1
fastqc    --version

## 0 · Download the real data

In [ ]:
%%bash
set -e
mkdir -p data && cd data
wget -q https://raw.githubusercontent.com/hanyingjhuang/ngs-sickle-tutorial/main/data/ref.fa
wget -q https://raw.githubusercontent.com/hanyingjhuang/ngs-sickle-tutorial/main/data/reads.fq
ls -la
echo
echo "=== reference ==="
head -1 ref.fa
echo "reference length: $(grep -v '^>' ref.fa | tr -d '\n' | wc -c) bp"
echo
echo "=== reads ==="
echo "read count: $(( $(wc -l < reads.fq) / 4 ))"

## 1 · Inspect raw reads
Each read is 4 lines: `@name` · sequence · `+` · quality (Phred+33).

In [ ]:
!head -8 data/reads.fq

Visualize the first read with bases coloured and quality bars below:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('data/reads.fq') as f:
    lines = [next(f).rstrip() for _ in range(8)]
for r in range(2):
    name, seq, _, qual = lines[r*4:r*4+4]
    Q = [ord(c)-33 for c in qual]
    fig, ax = plt.subplots(figsize=(14, 1.2))
    cmap = {'A':'#1a7f37','T':'#b1272d','C':'#0969da','G':'#9a6700','N':'#777'}
    for i, b in enumerate(seq):
        ax.text(i, 1.3, b, ha='center', va='center', color=cmap.get(b,'#000'), family='monospace', fontsize=9)
    ax.bar(range(len(Q)), Q, color=['#1a7f37' if q>=30 else '#9a6700' if q>=20 else '#bf6b00' if q>=10 else '#b1272d' for q in Q], width=0.9)
    ax.set_xlim(-0.5, len(seq)-0.5); ax.set_ylim(0, 42)
    ax.set_yticks([0,20,30,40]); ax.set_xticks([])
    ax.set_ylabel('Phred Q'); ax.set_title(name, fontsize=9, loc='left')
    for spine in ['top','right']: ax.spines[spine].set_visible(False)
    plt.tight_layout(); plt.show()

## 2 · QC with FastQC
FastQC scans the FASTQ for per-base quality, GC content, adapter contamination, etc.

In [ ]:
%%bash
mkdir -p qc
fastqc data/reads.fq -o qc 2>&1 | tail -3
ls qc/

Display the FastQC HTML report inline:

In [ ]:
from IPython.display import IFrame
IFrame('qc/reads_fastqc.html', width='100%', height=600)

## 3 · Adapter + quality trim with fastp
Drop adapters, trim 3' bases below Q20, filter reads that become too short.

In [ ]:
%%bash
fastp \
  -i data/reads.fq \
  -o data/trimmed.fq \
  -j data/qc.json -h data/qc.html \
  --cut_tail --cut_tail_window_size 4 --cut_tail_mean_quality 20 \
  2>&1 | tail -20

Plot per-base quality before vs after trimming, from the real `qc.json`:

In [ ]:
import json, matplotlib.pyplot as plt
qc = json.load(open('data/qc.json'))
before = qc['read1_before_filtering']['quality_curves']['mean']
after  = qc['read1_after_filtering']['quality_curves']['mean']
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axhspan(28, 40, color='#1a7f37', alpha=0.07)
ax.axhspan(20, 28, color='#9a6700', alpha=0.10)
ax.axhspan(0,  20, color='#b1272d', alpha=0.07)
ax.plot(before, '--', color='#b1272d', lw=1.6, label='before')
ax.plot(after,  '-',  color='#1a7f37', lw=1.6, label='after')
ax.set_xlabel('read position'); ax.set_ylabel('mean Phred quality')
ax.set_ylim(0, 41); ax.legend(loc='lower left', frameon=False)
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()

# Print a quick stats comparison
b, a = qc['summary']['before_filtering'], qc['summary']['after_filtering']
print(f"reads:  {b['total_reads']:,} -> {a['total_reads']:,}")
print(f"bases:  {b['total_bases']:,} -> {a['total_bases']:,}")
print(f"Q30:    {b['q30_rate']*100:.1f}% -> {a['q30_rate']*100:.1f}%")
print(f"GC:     {b['gc_content']*100:.1f}% -> {a['gc_content']*100:.1f}%")

## 4 · Align with BWA-MEM
Index the reference, then align trimmed reads.

In [ ]:
%%bash
bwa index data/ref.fa 2>&1 | tail -3
bwa mem -t 2 data/ref.fa data/trimmed.fq 2>data/bwa.log > data/aligned.sam
echo
echo "=== bwa log (last 10 lines) ==="
tail -10 data/bwa.log
echo
echo "=== aligned.sam first 4 lines ==="
head -4 data/aligned.sam | cut -c1-180

## 5 · Sort + index BAM
Convert SAM → sorted BAM, build a `.bai` index, summarise with `flagstat`.

In [ ]:
%%bash
samtools sort  data/aligned.sam -o data/aligned.bam
samtools index data/aligned.bam
samtools flagstat data/aligned.bam

Per-position depth across the reference:

In [ ]:
import subprocess, matplotlib.pyplot as plt
raw = subprocess.check_output(['samtools','depth','-a','data/aligned.bam'], text=True)
rows = [l.split('\t') for l in raw.strip().split('\n') if l]
pos = [int(r[1]) for r in rows]
dep = [int(r[2]) for r in rows]
RS334 = 5248232 - 5246000  # in our 1-based coord this is 2232
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.fill_between(pos, dep, color='#0a7c7e', alpha=0.4)
ax.plot(pos, dep, color='#0a7c7e', lw=0.9)
ax.axvline(RS334, color='#b1272d', lw=1, ls='--', label='rs334 (HbS)')
ax.set_xlabel('position on HBB region (chr11:5,246,001-5,250,000)')
ax.set_ylabel('read depth')
ax.legend(loc='upper right', frameon=False)
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()

## 6 · Pile up reads at each position
`mpileup` shows every reference base with the stack of read bases on top.

In [ ]:
%%bash
samtools mpileup -f data/ref.fa data/aligned.bam 2>/dev/null > data/pileup.txt
echo "pileup rows: $(wc -l < data/pileup.txt)"
echo
echo "=== rows around the sickle position (HBB:2232 / chr11:5,248,232) ==="
awk '$2 >= 2228 && $2 <= 2236' data/pileup.txt

Visual pileup window centred on the sickle site (`.` = match REF, lowercase = reverse strand):

In [ ]:
import matplotlib.pyplot as plt, re
from matplotlib.patches import Rectangle
from matplotlib.colors import to_rgba
rows = []
with open('data/pileup.txt') as f:
    for line in f:
        c, p, ref, dp, bases, qual = line.rstrip().split('\t')
        if 2218 <= int(p) <= 2246:
            rows.append((int(p), ref.upper(), int(dp), bases))
def parse_bases(refb, s):
    out = []; i = 0
    while i < len(s):
        ch = s[i]
        if ch == '^': i += 2; continue
        if ch == '$': i += 1; continue
        if ch in '+-':
            j = i+1
            while j < len(s) and s[j].isdigit(): j += 1
            n = int(s[i+1:j]); i = j + n; continue
        if ch in '.,': out.append(refb)
        elif ch.upper() in 'ACGTN': out.append(ch.upper())
        elif ch == '*': out.append('-')
        i += 1
    return out
cmap = {'A':'#1a7f37','T':'#b1272d','C':'#0969da','G':'#9a6700','N':'#777','-':'#777'}
max_stack = max(r[2] for r in rows)
fig, ax = plt.subplots(figsize=(13, max_stack*0.22 + 1.5))
for i, (p, ref, dp, bases) in enumerate(rows):
    ax.text(i, max_stack+1, ref, ha='center', va='center', color=cmap[ref], family='monospace', fontsize=11, weight='bold')
    calls = parse_bases(ref, bases)
    for k, b in enumerate(calls):
        match = b == ref
        ax.add_patch(Rectangle((i-0.4, max_stack-1-k), 0.8, 0.85, color=cmap.get(b,'#777'), alpha=0.18 if match else 0.92))
        if not match:
            ax.text(i, max_stack-1-k+0.42, b, ha='center', va='center', color='white', family='monospace', fontsize=8)
RS = next(i for i,(p,*_) in enumerate(rows) if p == 2232)
ax.add_patch(Rectangle((RS-0.5, -0.5), 1, max_stack+2.2, fill=False, ec='#b1272d', lw=1.5, ls='--'))
ax.set_xticks(range(len(rows))); ax.set_xticklabels([str(r[0]) for r in rows], rotation=90, fontsize=8)
ax.set_yticks([]); ax.set_xlim(-1, len(rows)); ax.set_ylim(-1, max_stack+2.5)
ax.set_title(f'pileup window — REF row at top, reads stacked below; red box = rs334 (sickle position)', fontsize=10)
for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout(); plt.show()

## 7 · Call variants → VCF
`bcftools mpileup` builds a likelihoods file; `bcftools call` emits a VCF.

In [ ]:
%%bash
bcftools mpileup -f data/ref.fa data/aligned.bam -Ou -o data/pile.bcf 2>&1 | tail -2
bcftools call -mv -Oz -o data/variants.vcf.gz data/pile.bcf 2>&1 | tail -2
bcftools index -f data/variants.vcf.gz
echo
echo "=== variants.vcf.gz ==="
bcftools view data/variants.vcf.gz 2>/dev/null | grep -v ^## | head -25

## 8 · Interpret — find the sickle cell variant
Pull out HBB:2232 (= chr11:5,248,232 GRCh37 = rs334), translate the codon, name the consequence.

In [ ]:
import subprocess, re
vcf = subprocess.check_output(['bcftools','view','data/variants.vcf.gz'], text=True)
rows = [l.split('\t') for l in vcf.splitlines() if l and not l.startswith('#')]

rs334 = next((r for r in rows if int(r[1]) == 2232), None)
if rs334 is None:
    print('rs334 was not called in this run')
else:
    fmt   = dict(zip(rs334[8].split(':'), rs334[9].split(':')))
    info  = dict(kv.split('=', 1) for kv in rs334[7].split(';') if '=' in kv)
    dp    = info.get('DP', '?')
    ad    = fmt.get('AD', '?,?').split(',')
    print('=' * 64)
    print(f"  rs334 (HbS) called from real HG02666 reads")
    print('=' * 64)
    print(f"  position    HBB:{rs334[1]}  =  chr11:5,248,232 (GRCh37)")
    print(f"  REF -> ALT  {rs334[3]} -> {rs334[4]}      (genomic forward strand)")
    print(f"  cDNA        c.20 A>T   (HBB on reverse strand; revcomp of T>A)")
    print(f"  protein     codon 7  GAG (Glu) -> GTG (Val)  =  p.Glu7Val")
    print(f"  genotype    {fmt['GT']}     (0/1 = heterozygous = sickle cell trait, HbAS)")
    print(f"  depth       {dp}    (REF reads = {ad[0]}, ALT reads = {ad[1]})")
    print(f"  QUAL        {rs334[5]}")
    print()
    print('  Phenotype: HbAS (sickle cell trait)')
    print('  - Heterozygous carrier; usually asymptomatic.')
    print('  - Confers protection against falciparum malaria.')
    print('  - Children of two HbAS carriers have 25% risk of HbSS (sickle disease).')
    print()
    print('  ClinVar VCV000015333 - pathogenic.   OMIM 603903.   dbSNP rs334.')

---
## What just happened

Every cell above ran a real Linux binary on real public sequencing data:

| step | tool | input → output |
| --- | --- | --- |
| inspect | `head` | reads.fq |
| QC | `fastqc` · `fastp` | reads.fq → qc.html, trimmed.fq |
| align | `bwa mem` | trimmed.fq + ref.fa → aligned.sam |
| sort/index | `samtools sort/index` | aligned.sam → aligned.bam + .bai |
| pileup | `samtools mpileup` | aligned.bam → pileup.txt |
| call | `bcftools mpileup/call` | aligned.bam → variants.vcf.gz |
| interpret | parsing in Python | variants.vcf.gz → rs334 / HbS |

**Try:** swap `data/reads.fq` for reads from a different 1000 Genomes sample. The pipeline runs unchanged; whether rs334 appears at HBB:2232 depends on whether that person carries the sickle allele.